# Практика · Дії у відео: що дає час і що його стирає

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі кліпи зошит малює формулами: відео з файлу тут
> не читається взагалі, бо пакетів `decord` і `av` у середовищі немає. Досить
> `torch`, `numpy`, `opencv-python`, `scikit-learn` і `matplotlib`. Готові
> відеомережі `torchvision` ми будуємо з `weights=None` — жодного байта з мережі.

> ⏱ Зошит навчає **пʼятдесят вісім мереж**: девʼятнадцять настройок по три зерна
> плюс одна окрема, ознаки якої ми потім читаємо трьома способами.
> Заміряно на чотирьох ядрах без відеокарти, в один потік: на вільній машині зошит
> друкує в кінці **193 секунди**, а коли на тій самій машині рахує щось іще — до
> **264**. Перевірка разом із запуском ядра додає зверху ще від 15 до 60 секунд.
> Тобто від трьох до пʼяти хвилин, і майже весь цей час — навчання. Майже весь цей час — навчання; у кінці зошит друкує розклад часу.

Що ми зробимо:

1. Збудуємо набір кліпів, у якому **один кадр марний за побудовою** — і доведемо
   це не словами, а трьома звірками.
2. Порівняємо чотири способи додати час: один кадр, кадри як канали і `Conv3d`
   із трьома різними читачами часу.
3. Переміщаємо кадри й подивимось, чи впаде точність — найдешевша перевірка
   чесності відеомоделі.
4. Перевіримо гіпотезу: **чи не стирає глобальне усереднення по часу рівно те,
   що часовий шар щойно знайшов**. Це головний замір теми.
5. Заміряємо, скільки дає окрема гілка руху (оптичний потік Farneback).
6. Знайдемо постановку, у якій `Conv3d` таки виграє в «кадрів як каналів».
7. Порахуємо ціну `(2+1)D` проти `3D` і побудуємо шість готових відеомереж.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

SIZE = 24                 # сторона кадру в пікселях
BALL_R = 2.6              # радіус кулі
RING_R = 7.0              # радіус кільцевої доріжки, по якій куля їздить
T_A = 12                  # довжина кліпу в наборі А
T_B = 16                  # довжина кліпу в наборі Б
SIG_LEN = 7               # скільки кадрів триває дія в наборі Б
TRAIN, TEST = 240, 300    # скільки кліпів навчальних і перевірних
SEEDS = (0, 1, 2)         # три зерна на кожну точку
EPOCHS_A, EPOCHS_B = 15, 25
BATCH, LR = 30, 3e-3
CHANCE = 1.0 / 3.0        # три класи, тож випадкове вгадування дає одну третину

NOTEBOOK_STARTED = time.time()
print("torch   ", torch.__version__)
print("cv2     ", cv2.__version__)
print("numpy   ", np.__version__)
print("потоків ", torch.get_num_threads())
print("випадкове вгадування: %.4f" % CHANCE)

## 1 · Чому звичайний відеонабір не годиться для нашого питання

Питання теми одне: **скільки дає час**. Щоб його поставити чесно, потрібен набір,
на якому один кадр не дає **нічого**. На справжніх відеонаборах це не так:
«плавання» впізнається по басейну, «гра на гітарі» — по гітарі. Модель, яка
дивиться на одну мить, там набирає багато, і різниця між 2D і 3D тоне в тому, що
обидві просто впізнають обстановку.

Тому набір ми будуємо самі — і будуємо так, щоб одна мить була марною
**за побудовою**, а не «на нашу думку».

### Пристрій набору А

Куля їздить по **кільцевій доріжці**. Її положення — це кут. Кожен кліп починається
з **випадкового кута** θ<sub>0</sub>, рівномірного на всьому колі. Дія — це слово з
дванадцяти кроків, де `+` означає крок за годинниковою стрілкою, а `−` проти.

Три дії відрізняються **тільки порядком кроків**. У кожної з них рівно шість `+` і
шість `−`, і всі три проходять ті самі дванадцять положень — різниця лише в тому,
коли саме куля туди потрапляє.

In [ ]:
STEP_WORDS = {
    "ривок":   "++++--+--+--",   # чотири кроки вперед підряд, далі рваний відхід
    "затинка": "+++-++---+--",   # три вперед, один назад, знову два вперед
    "сходи":   "++-++-++----",   # два вперед — один назад, і так тричі
}
NAMES = list(STEP_WORDS)


def positions(word):
    """Слово кроків -> послідовність положень на кільці, зсунута до нуля."""
    track = [0]
    for sign in word[:-1]:                    # останній крок веде вже за межі кліпу
        track.append(track[-1] + (1 if sign == "+" else -1))
    lowest = min(track)
    return [value - lowest for value in track]   # усі три дії живуть на тих самих висотах


PATTERNS = {name: positions(word) for name, word in STEP_WORDS.items()}

print("  дія       слово кроків   положення на кільці")
for name in NAMES:
    print("  %-9s %s   %s" % (name, STEP_WORDS[name], PATTERNS[name]))
print()
for name in NAMES:
    word = STEP_WORDS[name]
    print("  %-9s кроків «+»: %d, кроків «−»: %d"
          % (name, word.count("+"), word.count("-")))

### Звірка 1: три дії проходять той самий набір положень

Якщо мультимножини положень збігаються, то **усереднений по часу кадр** у трьох
дій однаковий до останнього біта. Це не косметична деталь: саме на ній тримається
головний замір теми.

In [ ]:
sorted_positions = {name: tuple(sorted(track)) for name, track in PATTERNS.items()}
for name in NAMES:
    print("  %-9s %s" % (name, list(sorted_positions[name])))

assert len(set(sorted_positions.values())) == 1, "набори положень розійшлися!"
print()
print("✅ усі три дії проходять той самий набір положень —")
print("   отже усереднений по часу кадр у них однаковий")

### Звірка 2: збігаються не тільки положення, а й пари й трійки

Згортка з ядром 3 по часу бачить не окремий кадр, а **вікно з трьох кадрів**. Тому
мало зрівняти окремі положення — треба зрівняти й статистику вікон. Рахуємо, скільки
разів у дії трапляється кожне вікно завдовжки `n` (вікно беремо відносним: віднімаємо
його перше значення, бо абсолютний кут усе одно випадковий).

In [ ]:
from collections import Counter


def window_counts(track, n):
    """Скільки разів трапляється кожне відносне вікно завдовжки n."""
    counts = Counter()
    for start in range(len(track) - n + 1):
        window = tuple(track[start + k] - track[start] for k in range(n))
        counts[window] += 1
    return tuple(sorted(counts.items()))


print("  n   однакові в усіх трьох дій?")
for n in range(1, 7):
    variants = {window_counts(PATTERNS[name], n) for name in NAMES}
    print("  %d   %s" % (n, "так" if len(variants) == 1 else "НІ — ось звідси видно різницю"))
print()
print("Отже: усе, що бачить одне вікно з трьох кадрів, у трьох дій однакове.")
print("Різниця зʼявляється лише на вікнах у чотири кадри й довших.")

### Малюємо кадри

Куля — це не «квадратик у клітинці», а мʼяка пляма: яскравість спадає від центра до
краю. Так кадр лишається схожим на справжній знімок, а не на таблицю.

In [ ]:
ROWS, COLUMNS = np.mgrid[0:SIZE, 0:SIZE]


def draw_frame(angle):
    """Один кадр: куля на кільці під заданим кутом."""
    center_x = SIZE / 2 + RING_R * math.cos(angle)
    center_y = SIZE / 2 + RING_R * math.sin(angle)
    distance = np.sqrt((COLUMNS - center_x) ** 2 + (ROWS - center_y) ** 2)
    # мʼякий край: у самому центрі 1.0, за радіусом кулі 0.0
    return np.clip(BALL_R + 0.5 - distance, 0.0, 1.0).astype(np.float32)


def make_clip_a(class_index, start_angle):
    """Кліп набору А: дія займає весь кліп і починається з першого кадру."""
    track = PATTERNS[NAMES[class_index]]
    clip = np.zeros((T_A, SIZE, SIZE), np.float32)
    for t in range(T_A):
        clip[t] = draw_frame(start_angle + 2 * math.pi * track[t] / 12.0)
    return clip


def make_set_a(count, seed):
    """Набір кліпів: класи по черзі, початковий кут випадковий."""
    rng = np.random.default_rng(seed)
    clips = np.zeros((count, T_A, SIZE, SIZE), np.float32)
    labels = np.zeros(count, np.int64)
    for i in range(count):
        labels[i] = i % 3
        clips[i] = make_clip_a(labels[i], rng.uniform(0, 2 * math.pi))
    order = rng.permutation(count)          # щоб класи не йшли рівним чергуванням
    return clips[order], labels[order]


train_a, train_y = make_set_a(TRAIN, 0)
test_a, test_y = make_set_a(TEST, 100)
print("навчальні кліпи:", train_a.shape, " перевірні:", test_a.shape)
print("класів у навчальній частині:", np.bincount(train_y))

### Звірка 3: усереднений по часу кадр у трьох дій збігається побітово

Беремо той самий початковий кут для всіх трьох дій і усереднюємо кожен кліп по часу.
Якщо різниця нульова — модель, яка усереднює час, не має шансів у принципі.

In [ ]:
angle = 0.7                                   # довільний, але той самий для трьох дій
mean_frames = [make_clip_a(c, angle).mean(axis=0) for c in range(3)]

print("  найбільша різниця між усередненими по часу кадрами:")
print("  ривок vs затинка: %.10f" % np.abs(mean_frames[0] - mean_frames[1]).max())
print("  ривок vs сходи:   %.10f" % np.abs(mean_frames[0] - mean_frames[2]).max())
assert np.abs(mean_frames[0] - mean_frames[1]).max() < 1e-6
assert np.abs(mean_frames[0] - mean_frames[2]).max() < 1e-6
print()
print("✅ усереднений по часу кадр у трьох дій той самий")

In [ ]:
fig, axes = plt.subplots(4, T_A, figsize=(T_A * 0.8, 3.6))
for c in range(3):
    clip = make_clip_a(c, angle)
    for t in range(T_A):
        axes[c][t].imshow(clip[t], cmap="magma", vmin=0, vmax=1)
        axes[c][t].axis("off")
    axes[c][0].set_ylabel(NAMES[c])
    axes[c][0].axis("on"); axes[c][0].set_xticks([]); axes[c][0].set_yticks([])
for t in range(T_A):
    axes[3][t].imshow(mean_frames[0], cmap="magma", vmin=0, vmax=1)
    axes[3][t].axis("off")
axes[0][0].set_title("кадр 1", fontsize=8, loc="left")
fig.suptitle("три дії покадрово; нижній рядок — усереднений по часу кадр (однаковий)", fontsize=9)
plt.tight_layout()
plt.show()
print("Кадри трьох дій різні, але кожен окремий кадр — це куля під якимось кутом,")
print("а кут узятий випадково. Нижній рядок і є доказ: час усереднили — різниця зникла.")

## 2 · Чотири способи подивитись на кліп

Кістяк у всіх однаковий: два згорткові шари, ширина 8 і 16, простір зменшуємо
пулінгом. Різниця лише в тому, **звідки береться час**:

| модель | що подають на вхід | що робить із часом |
|---|---|---|
| один кадр | середній кадр кліпу | нічого, часу немає |
| кадри як канали | усі кадри як канали 2D-згортки | кожен кадр привʼязаний до свого каналу |
| `Conv3d` | кліп цілком | ядро ковзає ще й по часу, ваги вздовж часу спільні |

А в `Conv3d` ми зробимо **три різні кінцівки** — три способи прочитати часову вісь
після згорток:

- **avg** — `AdaptiveAvgPool3d(1)`, тобто середнє по часу. Так зроблено в
  переважній більшості прикладів;
- **max** — максимум по часу: «чи трапилось це хоч раз»;
- **розгортка** — часову вісь не згортаємо взагалі, а розкладаємо в ознаки.

Простір усі три зводять однаково — середнім. Різниця **тільки** в часі, тож
порівняння чесне.

In [ ]:
class Net(nn.Module):
    """Один кістяк, чотири способи прочитати час."""

    def __init__(self, kind, readout="avg", t_len=T_A, width=(8, 16)):
        super().__init__()
        self.kind, self.readout, self.t_len = kind, readout, t_len
        first, second = width
        if kind == "3d":
            self.body = nn.Sequential(
                nn.Conv3d(1, first, 3, padding=1), nn.ReLU(),
                nn.MaxPool3d((1, 4, 4)),                 # час не чіпаємо, простір ділимо на 4
                nn.Conv3d(first, second, 3, padding=1), nn.ReLU(),
                nn.MaxPool3d((1, 2, 2)),
            )
        else:
            channels_in = t_len if kind == "stack" else 1
            self.body = nn.Sequential(
                nn.Conv2d(channels_in, first, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4),
                nn.Conv2d(first, second, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            )
        features = second * t_len if (kind == "3d" and readout == "flat") else second
        self.head = nn.Linear(features, 3)

    def trunk(self, x):
        """Ознаки після згорток: (кліпи, канали, час). Простір уже усереднено."""
        return self.body(x.unsqueeze(1)).mean(dim=(3, 4))

    def forward(self, x):                        # x: (кліпи, час, висота, ширина)
        if self.kind == "3d":
            h = self.trunk(x)                    # (кліпи, канали, час)
            if self.readout == "avg":
                h = h.mean(dim=2)                # середнє по часу
            elif self.readout == "max":
                h = h.max(dim=2).values          # максимум по часу
            else:
                h = h.flatten(1)                 # часову вісь розгортаємо в ознаки
        elif self.kind == "stack":
            h = self.body(x).mean(dim=(2, 3))
        else:
            middle = self.t_len // 2
            h = self.body(x[:, middle:middle + 1]).mean(dim=(2, 3))
        return self.head(h)


def weight_count(model):
    return sum(p.numel() for p in model.parameters())


for kind, readout, name in [("one", "-", "один кадр"), ("stack", "-", "кадри як канали"),
                            ("3d", "avg", "Conv3d + avg"), ("3d", "max", "Conv3d + max"),
                            ("3d", "flat", "Conv3d + розгортка")]:
    print("  %-20s ваг: %5d" % (name, weight_count(Net(kind, readout))))
print()
print("  перший шар «кадри як канали», Conv2d(12, 8, 3): %d ваг" % (12 * 8 * 9 + 8))
print("  перший шар Conv3d(1, 8, 3):                     %d ваг" % (1 * 8 * 27 + 8))
print("  крок кулі по кільцю: %.4f пікселя" % (2 * RING_R * math.sin(math.pi / 12)))

### Перевірка «наша реалізація = бібліотечна»

Перш ніж вірити числам, переконаємось, що ми правильно розуміємо, **що саме** робить
`Conv3d`. Напишемо тривимірну згортку циклами — так, як її означує формула, — і
звіримо з `F.conv3d`. Якщо збігається, значить наша картинка «ядро ковзає по часу»
не метафора, а буквальний опис.

In [ ]:
torch.manual_seed(0)
small_clip = torch.randn(1, 2, 5, 6, 6)          # (кліпи, канали, час, висота, ширина)
kernel = torch.randn(3, 2, 3, 3, 3)              # 3 вихідні канали, ядро 3×3×3
bias = torch.randn(3)

library = F.conv3d(small_clip, kernel, bias)     # без доповнення країв

# те саме циклами: для кожного вихідного каналу і кожного положення ядра
# складаємо добутки по каналу, часу, висоті й ширині
by_hand = torch.zeros_like(library)
for out_channel in range(3):
    for t in range(library.shape[2]):
        for y in range(library.shape[3]):
            for x in range(library.shape[4]):
                window = small_clip[0, :, t:t + 3, y:y + 3, x:x + 3]
                by_hand[0, out_channel, t, y, x] = (window * kernel[out_channel]).sum() + bias[out_channel]

difference = (by_hand - library).abs().max().item()
print("найбільша розбіжність із бібліотечною згорткою: %.9f" % difference)
assert difference < 1e-4, "розрахунок розійшовся!"
print("✅ збігається — Conv3d справді ковзає ядром і по часу теж")

## 3 · Табло 1: скільки дає час і чи справді модель ним користується

Кожну настройку навчаємо **на трьох зернах**. Поруч зі справжнім порядком кадрів
рахуємо другу колонку — **кадри перемішані**. Перемішування руйнує порядок, але
лишає той самий набір кадрів. Модель, яка справді читає рух, має впасти до
випадку; модель, яка знайшла обхідний шлях, лишиться високо.

Це найдешевша перевірка чесності відеомоделі, і повторити її можна на будь-якій
своїй.

In [ ]:
def train_and_test(model, clips_train, labels_train, clips_test, labels_test,
                   seed, epochs, batch=BATCH, lr=LR):
    """Навчання з нуля на заданому зерні; повертає частку правильних на перевірних."""
    torch.manual_seed(seed)
    for module in model.modules():                 # ваги перезапускаємо саме цим зерном
        if isinstance(module, (nn.Conv2d, nn.Conv3d, nn.Linear)):
            module.reset_parameters()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    x_train, y_train = torch.from_numpy(clips_train), torch.from_numpy(labels_train)
    x_test, y_test = torch.from_numpy(clips_test), torch.from_numpy(labels_test)
    shuffler = torch.Generator().manual_seed(seed)
    count = len(labels_train)
    for _ in range(epochs):
        order = torch.randperm(count, generator=shuffler)
        model.train()
        for start in range(0, count, batch):
            batch_index = order[start:start + batch]
            optimizer.zero_grad()
            loss_function(model(x_train[batch_index]), y_train[batch_index]).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        return (model(x_test).argmax(1) == y_test).float().mean().item()


def shuffle_time(clips, seed):
    """Кожному кліпу — своя перестановка кадрів. Набір кадрів той самий, порядку немає."""
    rng = np.random.default_rng(seed)
    mixed = clips.copy()
    for i in range(len(clips)):
        mixed[i] = clips[i][rng.permutation(clips.shape[1])]
    return mixed


def spread(values):
    return "%.4f ±%.4f" % (float(np.mean(values)), float(np.std(values)))


print("готово: навчання, перемішування кадрів і запис розкиду по зернах")
print("перевіримо перемішування на одному кліпі:")
example = shuffle_time(train_a[:1], 700)
print("  кадрів у кліпі до й після:", train_a[0].shape[0], example[0].shape[0])
print("  набір кадрів той самий:",
      np.allclose(np.sort(train_a[0].sum(axis=(1, 2))), np.sort(example[0].sum(axis=(1, 2)))))

In [ ]:
table_one = {}
started = time.time()

for kind, readout, name in [("one", "-", "один кадр"), ("stack", "-", "кадри як канали"),
                            ("3d", "avg", "Conv3d + avg по часу"),
                            ("3d", "max", "Conv3d + max по часу"),
                            ("3d", "flat", "Conv3d + розгортка часу")]:
    ordered, mixed = [], []
    for seed in SEEDS:
        ordered.append(train_and_test(Net(kind, readout), train_a, train_y,
                                      test_a, test_y, seed, EPOCHS_A))
        mixed_train = shuffle_time(train_a, 700 + seed)
        mixed_test = shuffle_time(test_a, 800 + seed)
        mixed.append(train_and_test(Net(kind, readout), mixed_train, train_y,
                                    mixed_test, test_y, seed, EPOCHS_A))
    table_one[name] = {"ваги": weight_count(Net(kind, readout)),
                       "порядок": ordered, "перемішано": mixed}

print("  модель                  |   ваг |  справжній порядок |  кадри перемішані")
for name, row in table_one.items():
    print("  %-23s | %5d |  %-17s |  %s"
          % (name, row["ваги"], spread(row["порядок"]), spread(row["перемішано"])))
print()
print("  випадкове вгадування: %.4f" % CHANCE)
print("  табло 1 рахувалось %.0f с" % (time.time() - started))

In [ ]:
print("  окремі зерна (справжній порядок):")
for name, row in table_one.items():
    print("  %-23s %s" % (name, ["%.4f" % v for v in row["порядок"]]))
print()
print("  окремі зерна (кадри перемішані):")
for name, row in table_one.items():
    print("  %-23s %s" % (name, ["%.4f" % v for v in row["перемішано"]]))

Три речі, які варто прочитати з табла:

1. **Один кадр дає рівно випадок.** Так і має бути — ми це збудували. На звичайному
   відеонаборі таку гарантію не отримати, і саме тому порівняння «2D проти 3D» на
   них так важко читати.
2. **Перемішування валить усе до випадку.** Отже моделі справді користуються
   порядком кадрів, а не якимось артефактом генератора.
3. **Кістяк один, а результат різний** — і різниця лежить не в згортках, а в тому,
   що стоїть після них. Ось на це й подивимось уважно.

## 4 · Головний замір: усереднення по часу стирає те, що знайшов часовий шар

Порівняння трьох окремо навчених мереж лишає лазівку: раптом `avg`-мережа просто
гірше навчилась? Приберемо лазівку. Візьмемо **одну** навчену мережу — ту, що дала
1.0000 з розгорткою часу, — і **її ж** ознаки прочитаємо трьома способами.

Ознаки після згорток мають форму `(кліпи, канали, час)`. На кожному з трьох
варіантів навчимо звичайну логістичну регресію. Згортки при цьому **ті самі**,
слово в слово, — міняється лише читач часу.

In [ ]:
strong = Net("3d", "flat")
strong_accuracy = train_and_test(strong, train_a, train_y, test_a, test_y, 0, EPOCHS_A)
print("мережа, ознаки якої беремо: Conv3d + розгортка часу, точність %.4f" % strong_accuracy)

with torch.no_grad():
    features_train = strong.trunk(torch.from_numpy(train_a)).numpy()   # (кліпи, канали, час)
    features_test = strong.trunk(torch.from_numpy(test_a)).numpy()
print("форма ознак:", features_train.shape, "— канали × час")


def linear_readout(train_matrix, test_matrix):
    """Проста лінійна модель поверх готових ознак."""
    model = LogisticRegression(max_iter=2000, multi_class="auto")
    model.fit(train_matrix, train_y)
    return model.score(test_matrix, test_y)


readouts = {
    "середнє по часу":      (features_train.mean(axis=2), features_test.mean(axis=2)),
    "максимум по часу":     (features_train.max(axis=2), features_test.max(axis=2)),
    "розгортка часу":       (features_train.reshape(len(train_y), -1),
                             features_test.reshape(len(test_y), -1)),
}
readout_scores = {}
print()
print("  читач часу поверх ОДНИХ І ТИХ САМИХ ознак:")
for name, (a, b) in readouts.items():
    readout_scores[name] = linear_readout(a, b)
    print("  %-20s %.4f" % (name, readout_scores[name]))
print()
print("  випадкове вгадування: %.4f" % CHANCE)

Ось і відповідь. Згортки знайшли все, що треба: розгорнувши час, проста лінійна
модель поверх них розділяє класи. Ті самі числа, стиснуті **середнім по часу**,
класів не розділяють узагалі.

І це не збіг обставин, а наслідок побудови набору: у трьох дій **однаковий набір
положень і однакова статистика вікон із трьох кадрів**. Усе, що можна дізнатись,
усереднивши по часу ознаки з коротким полем зору, у трьох дій однакове.

`AdaptiveAvgPool3d(1)`, який стоїть у кінці більшості прикладів із `Conv3d`, робить
рівно це: усереднює по часу. Часовий шар при цьому працює як слід — знецінює його
те, що стоїть після нього.

## 5 · Два потоки: вигляд окремо, рух окремо

Історично першою вдалою відеоархітектурою був **двопотоковий** підхід: одна гілка
дивиться на картинку, друга — на **оптичний потік**, тобто на поле зсувів між
сусідніми кадрами. Потік ми беремо той самий, що вводить
[тема 34](../34-video-tracking/lecture.html): `cv2.calcOpticalFlowFarneback`.

Заміряємо три варіанти: тільки вигляд, тільки рух, обидві гілки разом.

In [ ]:
def flow_stack(clips):
    """Оптичний потік між сусідніми кадрами: 2·(T−1) карт на кліп."""
    count, t_len, height, width = clips.shape
    flows = np.zeros((count, 2 * (t_len - 1), height, width), np.float32)
    for i in range(count):
        for t in range(t_len - 1):
            first = (np.clip(clips[i, t], 0, 1) * 255).astype(np.uint8)
            second = (np.clip(clips[i, t + 1], 0, 1) * 255).astype(np.uint8)
            field = cv2.calcOpticalFlowFarneback(first, second, None,
                                                 0.5, 3, 7, 3, 5, 1.2, 0)
            flows[i, 2 * t] = field[..., 0]        # зсув по горизонталі
            flows[i, 2 * t + 1] = field[..., 1]    # зсув по вертикалі
    return flows


started = time.time()
train_flow, test_flow = flow_stack(train_a), flow_stack(test_a)
middle = T_A // 2
train_look = train_a[:, middle:middle + 1].copy()
test_look = test_a[:, middle:middle + 1].copy()
print("потік порахований за %.1f с; карт на кліп: %d" % (time.time() - started, train_flow.shape[1]))
print("найбільший зсув у полі потоку: %.4f пікселя" % np.abs(train_flow).max())

In [ ]:
def branch(channels_in, first=8, second=16):
    return nn.Sequential(
        nn.Conv2d(channels_in, first, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4),
        nn.Conv2d(first, second, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    )


class TwoStream(nn.Module):
    """Дві гілки: вигляд і рух. Кожну можна вимкнути."""

    def __init__(self, use_look, use_move, flow_channels):
        super().__init__()
        self.use_look, self.use_move = use_look, use_move
        features = 0
        if use_look:
            self.look = branch(1); features += 16
        if use_move:
            self.move = branch(flow_channels); features += 16
        self.head = nn.Linear(features, 3)

    def forward(self, picture, flow):
        parts = []
        if self.use_look:
            parts.append(self.look(picture).mean(dim=(2, 3)))
        if self.use_move:
            parts.append(self.move(flow).mean(dim=(2, 3)))
        return self.head(torch.cat(parts, 1))


def train_two(model, seed, epochs=EPOCHS_A):
    torch.manual_seed(seed)
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            module.reset_parameters()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_function = nn.CrossEntropyLoss()
    x_look, x_move = torch.from_numpy(train_look), torch.from_numpy(train_flow)
    y = torch.from_numpy(train_y)
    shuffler = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        order = torch.randperm(len(train_y), generator=shuffler)
        model.train()
        for start in range(0, len(train_y), BATCH):
            k = order[start:start + BATCH]
            optimizer.zero_grad()
            loss_function(model(x_look[k], x_move[k]), y[k]).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        guess = model(torch.from_numpy(test_look), torch.from_numpy(test_flow)).argmax(1)
    return (guess == torch.from_numpy(test_y)).float().mean().item()


table_two = {}
for use_look, use_move, name in [(True, False, "лише вигляд"),
                                 (False, True, "лише рух"),
                                 (True, True, "два потоки")]:
    scores = [train_two(TwoStream(use_look, use_move, train_flow.shape[1]), s) for s in SEEDS]
    table_two[name] = {"ваги": weight_count(TwoStream(use_look, use_move, train_flow.shape[1])),
                       "точність": scores}

print("  гілка          |   ваг |  точність")
for name, row in table_two.items():
    print("  %-14s | %5d |  %s" % (name, row["ваги"], spread(row["точність"])))
print()
gain = np.mean(table_two["лише рух"]["точність"]) - np.mean(table_two["лише вигляд"]["точність"])
print("  гілка руху додає до гілки вигляду: %+.4f" % gain)

На нашому наборі гілка руху не «додає трохи» — вона **несе все**. Гілка вигляду
сама по собі не може нічого, бо ми так і збудували набір; разом із рухом вона нічого
не псує, але й не додає.

На справжніх наборах співвідношення інше: вигляд там сильний (басейн, гітара,
кухня), і рух додає одиниці відсотків. Наш замір показує другий бік тієї самої
монети: **коли вигляд вимкнено за побудовою, весь результат тримається на русі**.

## 6 · Табло 3: коли поділ ваг уздовж часу таки окупається

У наборі А дія займає весь кліп і починається з першого кадру. Там «кадри як канали»
працюють бездоганно: кожен кадр має свій канал, і мережі досить запамʼятати, що на
якому каналі.

Тепер зробимо набір Б: кліп на шістнадцять кадрів, дія триває сім і починається
**в довільний момент**. Куля до дії й після неї стоїть на місці. Дії три, і всі
повертають кулю точно туди, звідки вона почала, — тож ані перший, ані останній кадр
класу не видають.

`Conv3d` ділить ваги вздовж часу: те саме ядро дивиться на кожне положення в часі.
Саме тут за це має бути винагорода.

In [ ]:
SIGNATURES = {
    "гойдалка":  "+++---",     # три кроки вперед, три назад
    "тремтіння": "+-+-+-",     # дрібне тремтіння на місці
    "відкат":    "---+++",     # спершу назад, потім уперед
}
SIG_NAMES = list(SIGNATURES)


def signature_track(word):
    track = [0]
    for sign in word:
        track.append(track[-1] + (1 if sign == "+" else -1))
    return track                                  # сім значень, перше й останнє нулі


SIG_TRACKS = {name: signature_track(word) for name, word in SIGNATURES.items()}
for name in SIG_NAMES:
    print("  %-10s %s  ->  %s" % (name, SIGNATURES[name], SIG_TRACKS[name]))
print()
print("  усі три дії починаються й закінчуються в тій самій точці:",
      all(track[0] == 0 and track[-1] == 0 for track in SIG_TRACKS.values()))
print("  місць, де дія може початись: %d" % (T_B - SIG_LEN + 1))

In [ ]:
def make_clip_b(class_index, start_angle, begins_at):
    """Кліп набору Б: куля стоїть, потім робить дію, потім знову стоїть."""
    track = SIG_TRACKS[SIG_NAMES[class_index]]
    clip = np.zeros((T_B, SIZE, SIZE), np.float32)
    for t in range(T_B):
        step = t - begins_at
        offset = track[step] if 0 <= step < SIG_LEN else 0
        # крок дії робимо вдвічі більшим за крок набору А, щоб рух було видно
        clip[t] = draw_frame(start_angle + 2 * math.pi * offset * 2 / 12.0)
    return clip


def make_set_b(count, seed, anywhere):
    rng = np.random.default_rng(seed)
    clips = np.zeros((count, T_B, SIZE, SIZE), np.float32)
    labels = np.zeros(count, np.int64)
    middle_start = (T_B - SIG_LEN) // 2
    for i in range(count):
        labels[i] = i % 3
        begins_at = int(rng.integers(0, T_B - SIG_LEN + 1)) if anywhere else middle_start
        clips[i] = make_clip_b(labels[i], rng.uniform(0, 2 * math.pi), begins_at)
    order = rng.permutation(count)
    return clips[order], labels[order]


table_three = {}
started = time.time()
for anywhere in (False, True):
    column = "у довільний момент" if anywhere else "посередині кліпу"
    clips_train, labels_train = make_set_b(TRAIN, 0, anywhere)
    clips_test, labels_test = make_set_b(TEST, 100, anywhere)
    for kind, readout, name in [("one", "-", "один кадр"),
                                ("stack", "-", "кадри як канали"),
                                ("3d", "max", "Conv3d + max по часу")]:
        scores = [train_and_test(Net(kind, readout, t_len=T_B), clips_train, labels_train,
                                 clips_test, labels_test, seed, EPOCHS_B) for seed in SEEDS]
        table_three[(column, name)] = scores

print("  модель                  |  дія посередині  |  дія в довільний момент")
for name in ["один кадр", "кадри як канали", "Conv3d + max по часу"]:
    print("  %-23s |  %-15s |  %s"
          % (name, spread(table_three[("посередині кліпу", name)]),
             spread(table_three[("у довільний момент", name)])))
print()
print("  окремі зерна, дія в довільний момент:")
for name in ["кадри як канали", "Conv3d + max по часу"]:
    print("  %-23s %s" % (name, ["%.4f" % v for v in table_three[("у довільний момент", name)]]))
print()
for name in ["кадри як канали", "Conv3d + max по часу"]:
    drop = (np.mean(table_three[("посередині кліпу", name)])
            - np.mean(table_three[("у довільний момент", name)]))
    print("  %-23s втрачає від зсуву дії: %.4f" % (name, drop))
print("  табло 3 рахувалось %.0f с" % (time.time() - started))

## 7 · Ціна тривимірної згортки і розклад `(2+1)D`

`Conv3d` дорога не так вагами, як обчисленнями: ядро `3×3×3` має **27** множень на
кожну пару каналів проти **9** у `3×3`. Розклад `(2+1)D` розриває це ядро надвоє:
спершу просторова згортка `1×3×3`, потім часова `3×1×1`, а між ними — проміжні
канали.

Порахуємо ваги руками й звіримо з `torch`.

In [ ]:
def conv3d_weights(channels_in, channels_out):
    return channels_in * channels_out * 27 + channels_out


def r2plus1d_weights(channels_in, channels_out, middle):
    spatial = channels_in * middle * 9 + middle       # 1×3×3
    temporal = middle * channels_out * 3 + channels_out   # 3×1×1
    return spatial + temporal


def equal_cost_middle(channels_in, channels_out):
    """Проміжний розмір, за якого ваг у розкладі рівно стільки ж, скільки в 3D."""
    return (27 * channels_in * channels_out) // (9 * channels_in + 3 * channels_out)


print("  Conv3d(64, 64, 3)          %8d" % conv3d_weights(64, 64))
print("  (1×3×3)+(3×1×1), M = 128   %8d" % r2plus1d_weights(64, 64, 128))
print("  Conv2d(64, 64, 3)          %8d" % (64 * 64 * 9 + 64))
print()
star = equal_cost_middle(64, 64)
print("  розвʼязок для 64→64 без зсувів: %d ділимо на %d, виходить %d"
      % (27 * 64 * 64, 9 * 64 + 3 * 64, star))
print("  проміжний розмір «нарівні за вагами» для 64→64: M* = %d" % star)
print("  (1×3×3)+(3×1×1), M = M*    %8d" % r2plus1d_weights(64, 64, star))
print()
print("  Звірка з torch:")
built_3d = nn.Conv3d(64, 64, 3)
built_spatial = nn.Conv3d(64, star, (1, 3, 3))
built_temporal = nn.Conv3d(star, 64, (3, 1, 1))
print("  nn.Conv3d(64, 64, 3):                     %8d" % weight_count(built_3d))
print("  nn.Conv3d(64,M*,(1,3,3)) + (M*,64,(3,1,1)): %8d"
      % (weight_count(built_spatial) + weight_count(built_temporal)))
assert weight_count(built_3d) == conv3d_weights(64, 64), "формула розійшлася з torch!"
assert weight_count(built_spatial) + weight_count(built_temporal) == r2plus1d_weights(64, 64, star)
print("✅ формули збігаються з torch")

Ваг у розкладі стільки ж, а нелінійностей **удвічі більше**: `ReLU` стоїть і після
просторової згортки, і після часової. Саме в цьому автори `R(2+1)D` бачили виграш —
не в економії, а в тому, що та сама кількість ваг дає глибшу нелінійну функцію.

## 8 · Готові відеомережі: будуються офлайн

`torchvision` уміє збудувати шість відеоархітектур без жодного байта з мережі —
досить `weights=None`. Порахуємо їхні ваги й перевіримо, на якому вході вони
рахуються.

In [ ]:
import torchvision.models.video as video_models

print("  архітектура    параметрів")
sizes = {}
for name in ["s3d", "mc3_18", "swin3d_t", "r2plus1d_18", "r3d_18", "mvit_v2_s"]:
    model = video_models.__dict__[name](weights=None)
    sizes[name] = weight_count(model)
    print("  %-12s %11d" % (name, sizes[name]))
print()
print("  найлегша: %s, найважча: %s" % (min(sizes, key=sizes.get), max(sizes, key=sizes.get)))
print("  різниця: у %.1f раза" % (max(sizes.values()) / min(sizes.values())))

In [ ]:
# r3d_18 на класичному вході 16 кадрів 112×112 рахується
net = video_models.r3d_18(weights=None).eval()
with torch.no_grad():
    out = net(torch.zeros(1, 3, 16, 112, 112))
print("r3d_18 на (1, 3, 16, 112, 112) ->", tuple(out.shape))

# а s3d на тому самому вході падає: у нього перше ядро 7×7 і забагато зменшень
net = video_models.s3d(weights=None).eval()
try:
    with torch.no_grad():
        net(torch.zeros(1, 3, 16, 112, 112))
    print("s3d на 112×112: порахувався")
except RuntimeError as error:
    print("s3d на 112×112 ПАДАЄ:", str(error)[:90])
with torch.no_grad():
    out = net(torch.zeros(1, 3, 16, 224, 224))
print("s3d на (1, 3, 16, 224, 224) ->", tuple(out.shape))
print()
print("Мораль: «найлегша модель» не означає «найневибагливіша до входу».")

In [ ]:
print("увесь зошит виконувався %.0f с" % (time.time() - NOTEBOOK_STARTED))

## Що робити далі

### 🟢 Рівень 1 — База

Додай до табла 1 четверту кінцівку для `Conv3d`: **середнє по перших шести кадрах і
середнє по останніх шести, склеєні разом**. Це грубий компроміс між усередненням і
розгорткою.

**Зроблено, якщо:** у таблиці зʼявився рядок із трьома зернами, і ти словами
пояснив, чому він став між `avg` і розгорткою (або чому не став).

### 🟡 Рівень 2 — Плюс

Візьми набір Б і поміняй у ньому одну річ: хай дія **не повертає** кулю в початкову
точку (наприклад, `гойдалка` стане `++++--`). Проведи ті самі три моделі.

**Зроблено, якщо:** ти показав числами, що «кадри як канали» піднялись, і пояснив,
за яку саме підказку вони вхопились.

### 🔴 Рівень 3 — Виклик

Збудуй набір, у якому **однакові й вікна з чотирьох кадрів** — тобто дії
розрізняються лише вікнами завдовжки пʼять і більше. Підказка: перебери всі слова
з шести `+` і шести `−` і згрупуй їх за статистикою вікон, як це зроблено у
звірці 2.

**Зроблено, якщо:** ти знайшов таку трійку дій (або довів перебором, що при
дванадцяти кроках її немає), і заміряв на ній `Conv3d` із розгорткою часу.